In [ ]:
from google.colab import drive
drive.mount('/content/drive', )

In [ ]:
import os
import time

PROJECT_DIR = "/content/drive/MyDrive/SleepStageClassificationProject"
DATASET_DIR = os.path.join(PROJECT_DIR, "dataset")
HMC_DIR = os.path.join(DATASET_DIR, "HMC")
os.makedirs(HMC_DIR, exist_ok=True)

%cd $HMC_DIR

In [ ]:
!wget -q -N https://physionet.org/files/hmc-sleep-staging/1.1/RECORDS

with open("RECORDS", "r") as f:
    raw_lines = [line.strip() for line in f if line.strip()]

subjects = [line.replace("recordings/", "").replace(".edf", "") for line in raw_lines]
print(f"Total subjects: {len(subjects)}")
print(f"First 5: {subjects[:5]}")

In [ ]:
#download
import os
import time
import json

BASE_URL = "https://physionet.org/files/hmc-sleep-staging/1.1/recordings"
CHECKPOINT_FILE = "download_checkpoint.json"

def load_checkpoint():
    if os.path.exists(CHECKPOINT_FILE):
        with open(CHECKPOINT_FILE, 'r') as f:
            return json.load(f)
    return {"completed": [], "failed_files": []}

def save_checkpoint(data):
    with open(CHECKPOINT_FILE, 'w') as f:
        json.dump(data, f)

def is_file_complete(fname):
    if not os.path.exists(fname):
        return False
    size = os.path.getsize(fname)
    if fname.endswith(".edf") and "sleepscoring" not in fname:
        return size > 10_000_000
    return size > 1000

def download_with_retry(url, filename, max_retries=3, retry_delay=5):
    for attempt in range(1, max_retries + 1):
        if attempt > 1 and os.path.exists(filename):
            if not is_file_complete(filename):
                os.remove(filename)

        try:
            result = !wget -q --timeout=30 -O {filename} {url} 2>&1

            if is_file_complete(filename):
                return True
            else:
                if attempt < max_retries:
                    time.sleep(retry_delay)
        except:
            if attempt < max_retries:
                time.sleep(retry_delay)

    return False

checkpoint = load_checkpoint()
completed_subjects = set(checkpoint.get("completed", []))
failed_files = checkpoint.get("failed_files", [])

print("="*70)
print("📥 DOWNLOAD & REPAIR PHASE")
print("="*70)
if completed_subjects:
    print(f"📍 Resuming from checkpoint: {len(completed_subjects)}/{len(subjects)} subjects done\n")
else:
    print(f"📍 Starting fresh: {len(subjects)} subjects to download\n")

start_time = time.time()
batch_count = 0

for i, subject in enumerate(subjects, 1):
    if subject in completed_subjects:
        print(f"[{i:3d}/{len(subjects)}] {subject}: ⊘ skip")
        continue

    edf_file = f"{subject}.edf"
    scoring_edf_file = f"{subject}_sleepscoring.edf"
    txt_file = f"{subject}_sleepscoring.txt"

    files_to_download = [
        (edf_file, f"{BASE_URL}/{edf_file}"),
        (scoring_edf_file, f"{BASE_URL}/{scoring_edf_file}"),
        (txt_file, f"{BASE_URL}/{txt_file}")
    ]

    checks = []
    subject_complete = True

    for fname, url in files_to_download:
        if is_file_complete(fname):
            checks.append("✓")
        else:
            if download_with_retry(url, fname):
                checks.append("✓")
            else:
                checks.append("✗")
                subject_complete = False
                if fname not in failed_files:
                    failed_files.append(fname)


    if subject_complete:
        completed_subjects.add(subject)

    elapsed_min = (time.time() - start_time) / 60
    status = " ".join(checks)
    print(f"[{i:3d}/{len(subjects)}] {subject}: {status}  ({elapsed_min:6.1f} min)")

    batch_count += 1
    if batch_count % 5 == 0:
        save_checkpoint({"completed": list(completed_subjects), "failed_files": failed_files})
        print(f"   💾 Checkpoint saved ({len(completed_subjects)}/{len(subjects)})")

save_checkpoint({"completed": list(completed_subjects), "failed_files": failed_files})

print("\n" + "="*70)
print("✅ Download phase complete!")
print(f"   ✓ Complete subjects: {len(completed_subjects)}/{len(subjects)}")
print(f"   ✗ Files to repair: {len(set(failed_files))}")
print("="*70)

if failed_files:
    print(f"\n🔧 Repairing {len(set(failed_files))} incomplete files...\n")

    unique_failed = list(set(failed_files))
    fixed = 0
    still_broken = []

    for i, fname in enumerate(unique_failed, 1):
        url = f"{BASE_URL}/{fname}"

        if os.path.exists(fname):
            os.remove(fname)

        print(f"[{i:3d}/{len(unique_failed)}] ↻ {fname:35}", end=" ")

        if download_with_retry(url, fname, max_retries=5):
            print("✓")
            fixed += 1
        else:
            print("✗")
            still_broken.append(fname)

    print("\n" + "="*70)
    print("🔧 REPAIR SUMMARY")
    print("="*70)
    print(f"   ✓ Fixed: {fixed}/{len(unique_failed)}")
    print(f"   ✗ Still broken: {len(still_broken)}")

    if still_broken:
        print(f"\n⚠️  Failed to repair ({len(still_broken)}):")
        for f in still_broken[:10]:
            print(f"   - {f}")
        if len(still_broken) > 10:
            print(f"   ... and {len(still_broken)-10} more")
        print("\n💡 Try again later or check PhysioNet server")
    else:
        print("\n🎉 All files repaired!")
else:
    print("\n✨ No files to repair!")

print("\n" + "="*70)
print("📋 FINAL VERIFICATION")
print("="*70)

complete_subjects = []
incomplete_subjects = []

for subject in subjects:
    files_ok = all(
        os.path.exists(f"{subject}{suffix}") and is_file_complete(f"{subject}{suffix}")
        for suffix in [".edf", "_sleepscoring.edf", "_sleepscoring.txt"]
    )

    if files_ok:
        complete_subjects.append(subject)
    else:
        incomplete_subjects.append(subject)

print(f"\n✓ Complete subjects: {len(complete_subjects)}/{len(subjects)}")
print(f"✗ Incomplete subjects: {len(incomplete_subjects)}/{len(subjects)}")

if incomplete_subjects:
    print(f"\n  Incomplete ({len(incomplete_subjects)}):")
    for s in incomplete_subjects[:10]:
        print(f"    - {s}")
    if len(incomplete_subjects) > 10:
        print(f"    ... and {len(incomplete_subjects)-10} more")
    print("\n  💾 Checkpoint saved - can resume by running this cell again")
else:
    print("\n🚀 Ready for next section!")
    if os.path.exists(CHECKPOINT_FILE):
        os.remove(CHECKPOINT_FILE)
        print("   ✓ Checkpoint cleaned up")

In [ ]:
print("\n📦 Copying to Google Drive...")
!cp -n {LOCAL_DIR}/*.edf {LOCAL_DIR}/*.txt {LOCAL_DIR}/RECORDS "{HMC_DIR}/" 2>/dev/null
print("✅ Copy to Drive complete!")

In [ ]:
edf_files = [f for f in os.listdir(HMC_DIR) if f.endswith('.edf') and 'sleepscoring' not in f]
scoring_edf_files = [f for f in os.listdir(HMC_DIR) if f.endswith('_sleepscoring.edf')]
txt_files = [f for f in os.listdir(HMC_DIR) if f.endswith('sleepscoring.txt')]

print(f"📊 PSG EDF files: {len(edf_files)}")
print(f"📊 Scoring EDF files: {len(scoring_edf_files)}")
print(f"📊 Scoring TXT files: {len(txt_files)}")

complete = sum(1 for s in subjects
               if os.path.exists(os.path.join(HMC_DIR, f"{s}.edf"))
               and os.path.exists(os.path.join(HMC_DIR, f"{s}_sleepscoring.edf"))
               and os.path.exists(os.path.join(HMC_DIR, f"{s}_sleepscoring.txt")))
print(f"\nครบทั้ง 3 ไฟล์บน Drive: {complete}/{len(subjects)}")


In [ ]:
!pip install mne -q

import mne

first_subject = subjects[0]
edf_path_full = os.path.join(HMC_DIR, f"{first_subject}.edf")

print(f"Loading {first_subject}...")
raw = mne.io.read_raw_edf(edf_path_full, preload=False, verbose=False)

print(f"\n📋 Dataset Info:")
print(f"Duration: {raw.n_times / raw.info['sfreq'] / 3600:.1f} hours")
print(f"Sampling rate: {raw.info['sfreq']} Hz")
print(f"\n📡 Available Channels ({len(raw.ch_names)}):")
for i, ch in enumerate(raw.ch_names, 1):
    print(f"  {i:2d}. {ch}")

OPTIMAL_CHANNELS = {
    'EEG': ['EEG F4-M1', 'EEG C4-M1', 'EEG O2-M1', 'EEG C3-M2'],
    'EOG': ['EOG E1-M2', 'EOG E2-M2'],
    'EMG': ['EMG chin'],
    'ECG': ['ECG']
}

CHANNELS_TO_USE = []
for group, channels in OPTIMAL_CHANNELS.items():
    CHANNELS_TO_USE.extend(channels)

print("\n🎯 Optimal Channel Configuration:")
print("="*50)
for group, channels in OPTIMAL_CHANNELS.items():
    print(f"\n{group} ({len(channels)} channels):")
    for ch in channels:
        status = "✓" if ch in raw.ch_names else "✗"
        print(f"  {status} {ch}")

raw_full = mne.io.read_raw_edf(edf_path_full, preload=True, verbose=False)
available_channels = [ch for ch in CHANNELS_TO_USE if ch in raw_full.ch_names]
raw_selected = raw_full.copy().pick(available_channels)

print(f"\n✅ Selected {len(available_channels)} channels")
print(f"Duration: {raw_selected.n_times / raw_selected.info['sfreq'] / 3600:.1f} hours")